## Part 2 of 6: Data Cleaning

Loads df and RAW_DOLLAR_COLS from notebooks/01_loading_data.ipynb.

See `notebooks/README.md` for the full run order.

In [ ]:
# Import Python libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_recall_curve, precision_score, recall_score)
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Libraries have been imported!")

In [ ]:
# Load state saved by the previous notebook
import joblib
_state = joblib.load("_state/01_state.joblib")
globals().update(_state)
print(f"Loaded {len(_state)} objects: {sorted(_state)}")


In [ ]:
# Check if there are any missing values
nulls = df.isnull().sum()
print("Total missing values:", nulls.sum())
assert nulls.sum() == 0, f"Error: Missing values found!\n{nulls[nulls > 0]}"

# Count number of unique and duplicate firms
n_firms = df["company_name"].nunique()
print(f"Unique firms: {n_firms:,}")
dupes = df.duplicated(["company_name", "year"]).sum()
print(f"Duplicate (firm, year) rows: {dupes}")
assert dupes == 0, "Error: Duplicate (firm, year) tuples found!"

# Sort dataset by company name and year
df = df.sort_values(["company_name", "year"]).reset_index(drop=True)
print("\n")

# Is a firm listed as 'alive' before it goes bankrupt, or is it 'failed' all the way?
labels_per_firm = df.groupby("company_name")["status_label"].nunique()
print(f"Firms with more than one distinct label: "
      f"{(labels_per_firm > 1).sum()} of {n_firms:,}")

# The label is constant per firm! Every historical row of a failed firm
# says 'failed'. We should relabel this and only keep 'failed' on
# the firm's final observed year, and relabel all earlier years as 'alive'.
final_year = df.groupby("company_name")["year"].transform("max")
is_final_row = df["year"] == final_year
df["target"] = ((df["status_label"] == "failed") & is_final_row).astype(int)
relabeled = ((df["status_label"] == "failed") & ~is_final_row).sum()
print(f"Relabeled {relabeled:,} pre-final rows of failed firms "
      f"to alive. Positive instances (bankrupt next year): {df['target'].sum()}")

# Verify that our relabeling worked
failed_firms = df.loc[df["status_label"] == "failed", "company_name"].unique()
rows_after_final = df[(df["company_name"].isin(failed_firms)) &
                      (df["year"] > final_year)]
print(f"Failed-firm rows appearing after final year: "
      f"{len(rows_after_final)}")
print("\n")

# Number of firms with year gaps (matters for trend features later)
year_gap = df.groupby("company_name")["year"].diff()
firms_with_gaps = df.loc[year_gap > 1, "company_name"].nunique()
print(f"Firms with gap years in their history: {firms_with_gaps} "
      f"(trend features will be masked across gaps)")

In [ ]:
# Save state for the next notebook
import joblib
from pathlib import Path
Path("_state").mkdir(exist_ok=True)
joblib.dump({
    "df": df, "RAW_DOLLAR_COLS": RAW_DOLLAR_COLS, "failed_firms": failed_firms
}, "_state/02_state.joblib")
